# Building a Product Recommendation Algorithm with Python

In [ ]:
# Load dataset
import pandas as pd
import janitor

df = pd.read_excel('../../data/raw/Online Retail.xlsx', sheet_name='Online Retail')
df = df.clean_names(case_type='snake')
df.head()

In [ ]:
# Filter out cancelled orders
df = df.loc[df['quantity'] > 0]

# Data Preparation

In [ ]:
df.loc[df['customer_id'].isna()].head()

In [ ]:
# Drop rows with missing customer IDs
df = df.dropna(subset=['customer_id'])

# Building a Customer-Item Matrix

In [ ]:
customer_item_matrix = df.pivot_table(
    index='customer_id', 
    columns='stock_code', 
    values='quantity',
    aggfunc='sum'
)

In [ ]:
# Review data
customer_item_matrix.loc[12481:].head()

In [ ]:
customer_item_matrix.shape

In [ ]:
customer_item_matrix = (customer_item_matrix > 0).astype(int)

In [ ]:
customer_item_matrix

# Colaborative Filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

user_user_sim_matrix = pd.DataFrame(
    cosine_similarity(customer_item_matrix)
)

In [ ]:
user_user_sim_matrix.head()

In [ ]:
user_user_sim_matrix.columns = customer_item_matrix.index

user_user_sim_matrix['customer_id'] = customer_item_matrix.index
user_user_sim_matrix = user_user_sim_matrix.set_index('customer_id')

In [ ]:
user_user_sim_matrix.head()

In [ ]:
user_user_sim_matrix.loc[12350.0].sort_values(ascending=False)

In [ ]:
customer_A = customer_item_matrix.loc[12350.0]
items_bought_by_A = set(customer_A[customer_A > 0].index)
items_bought_by_A

In [ ]:
items_bought_by_B = set(customer_item_matrix.loc[17935.0][customer_item_matrix.loc[17935.0] > 0].index)
items_bought_by_B

In [ ]:
items_to_recommend_to_B = items_bought_by_A - items_bought_by_B
items_to_recommend_to_B

In [ ]:
df.loc[
    df['stock_code'].isin(items_to_recommend_to_B), 
    ['stock_code', 'description']
].drop_duplicates().set_index('stock_code')

# Item-Based Collaborative Filtering and Product Recommendations

In [ ]:
item_item_sim_matrix = pd.DataFrame(cosine_similarity(customer_item_matrix.T))

In [ ]:
item_item_sim_matrix.columns = customer_item_matrix.T.index

item_item_sim_matrix['stock_code'] = customer_item_matrix.T.index
item_item_sim_matrix = item_item_sim_matrix.set_index('stock_code')

In [ ]:
item_item_sim_matrix

# Making Recommendations

In [ ]:
top_10_similar_items = list(
    item_item_sim_matrix\
        .loc[23166]\
        .sort_values(ascending=False)\
        .iloc[:10]\
    .index
)

In [ ]:
top_10_similar_items

In [ ]:
df.loc[
    df['stock_code'].isin(top_10_similar_items), 
    ['stock_code', 'description']
].drop_duplicates().set_index('stock_code').loc[top_10_similar_items]